# Reading Data from Parquet Files

This notebook demonstrates how to read and query market data stored as Parquet files using:

1. **DuckDB** - SQL queries on Parquet files
2. **Reader Repositories** - Convenient Python API for common queries
3. **PyArrow** - Direct Parquet access with predicate pushdown

## Prerequisites

You should have run `01_quickstart.ipynb` first to populate some sample data.

## Table of Contents

1. [Understanding Parquet Storage](#1-understanding-parquet-storage)
2. [Querying with DuckDB](#2-querying-with-duckdb)
3. [Using Reader Repositories](#3-using-reader-repositories)
4. [Advanced PyArrow Queries](#4-advanced-pyarrow-queries)
5. [Performance Comparison](#5-performance-comparison)

## 1. Understanding Parquet Storage

First, let's explore the Parquet file structure on disk.

In [ ]:
import os
from pathlib import Path
import duckdb
import pyarrow.dataset as ds
import pyarrow.compute as pc
import pandas as pd
from datetime import date, timedelta

# Import reader repositories
from dlt_ibapi.repositories import (
    EquityBarsReader,
    OptionBarsReader,
    OptionChainSnapshotReader,
)

# Data directory
data_dir = Path("../data")

print("✓ Libraries imported")

### Explore the directory structure

In [ ]:
def show_tree(directory, prefix="", max_depth=3, current_depth=0):
    """Display directory tree structure."""
    if current_depth >= max_depth:
        return
    
    items = sorted(Path(directory).iterdir(), key=lambda x: (not x.is_dir(), x.name))
    
    for i, item in enumerate(items):
        is_last = i == len(items) - 1
        current_prefix = "└── " if is_last else "├── "
        print(f"{prefix}{current_prefix}{item.name}")
        
        if item.is_dir():
            extension = "    " if is_last else "│   "
            show_tree(item, prefix + extension, max_depth, current_depth + 1)

if data_dir.exists():
    print("\nParquet Storage Structure:")
    print(f"{data_dir.name}/")
    show_tree(data_dir)
else:
    print(f"\n⚠️  Data directory not found: {data_dir}")
    print("   Please run 01_quickstart.ipynb first to generate sample data.")

### Inspect Parquet file metadata

In [ ]:
import pyarrow.parquet as pq

# Find a sample Parquet file
parquet_files = list(data_dir.rglob("*.parquet"))

if parquet_files:
    sample_file = parquet_files[0]
    print(f"\nInspecting: {sample_file.relative_to(data_dir.parent)}")
    
    # Read Parquet metadata
    parquet_file = pq.ParquetFile(sample_file)
    
    print(f"\n📊 File Statistics:")
    print(f"  Rows: {parquet_file.metadata.num_rows:,}")
    print(f"  Columns: {parquet_file.metadata.num_columns}")
    print(f"  File size: {sample_file.stat().st_size / 1024:.2f} KB")
    
    print(f"\n📋 Schema:")
    for field in parquet_file.schema:
        print(f"  - {field.name}: {field.type}")
else:
    print("\n⚠️  No Parquet files found. Run 01_quickstart.ipynb first.")

## 2. Querying with DuckDB

DuckDB provides fast SQL queries on Parquet files with Hive partitioning support.

### Basic queries

In [ ]:
# Connect to DuckDB in-memory
conn = duckdb.connect(":memory:")

# Query all equity bars
df = conn.execute("""
    SELECT *
    FROM parquet_scan('../data/stocks/**/*.parquet', hive_partitioning=true)
    LIMIT 5
""").df()

print("\nSample equity bars:")
df

### Using partition columns for filtering

In [ ]:
# Filter by date and symbol using partitions (very fast!)
df = conn.execute("""
    SELECT 
        symbol,
        time::DATE as date,
        open,
        high,
        low,
        close,
        volume
    FROM parquet_scan('../data/stocks/**/*.parquet', hive_partitioning=true)
    WHERE symbol = 'AAPL' 
      AND date >= CURRENT_DATE - INTERVAL '7 days'
    ORDER BY time
""").df()

print(f"\nAAPL bars from last 7 days: {len(df)} rows")
df.head(10)

### Aggregations and analytics

In [ ]:
# Calculate daily returns
returns_df = conn.execute("""
    WITH daily_prices AS (
        SELECT 
            symbol,
            time::DATE as date,
            close,
            LAG(close) OVER (PARTITION BY symbol ORDER BY time) as prev_close
        FROM parquet_scan('../data/stocks/**/*.parquet', hive_partitioning=true)
    )
    SELECT 
        symbol,
        date,
        close,
        ROUND(((close - prev_close) / prev_close * 100), 2) as daily_return_pct
    FROM daily_prices
    WHERE prev_close IS NOT NULL
    ORDER BY symbol, date DESC
""").df()

print("\nDaily returns (%):\n")
returns_df.head(10)

### Summary statistics per symbol

In [ ]:
stats_df = conn.execute("""
    SELECT 
        symbol,
        COUNT(*) as bar_count,
        MIN(time::DATE) as first_date,
        MAX(time::DATE) as last_date,
        ROUND(AVG(close), 2) as avg_close,
        ROUND(MIN(low), 2) as min_low,
        ROUND(MAX(high), 2) as max_high,
        ROUND(AVG(volume), 0) as avg_volume
    FROM parquet_scan('../data/stocks/**/*.parquet', hive_partitioning=true)
    GROUP BY symbol
    ORDER BY symbol
""").df()

print("\nSummary statistics per symbol:\n")
stats_df

## 3. Using Reader Repositories

The `dlt-ibapi` library provides convenient Python APIs for reading Parquet data.

### EquityBarsReader

In [ ]:
# Initialize equity reader
equity_reader = EquityBarsReader("../data", "stocks")

print("✓ EquityBarsReader initialized")
print(f"  Dataset: {equity_reader.dataset_name}")
print(f"  Destination: {equity_reader.destination_type}")

#### Get available symbols

In [ ]:
symbols = equity_reader.get_available_symbols(bar_size="1 day")
print(f"\nAvailable symbols: {symbols}")

#### Get date range for a symbol

In [ ]:
if symbols:
    symbol = symbols[0]
    min_date, max_date = equity_reader.get_date_range(symbol, "1 day")
    print(f"\n{symbol} date range:")
    print(f"  First bar: {min_date}")
    print(f"  Last bar: {max_date}")
    print(f"  Total days: {(max_date - min_date).days}")

#### Get bars for a symbol (with PyArrow optimization)

In [ ]:
if symbols:
    # Get bars using PyArrow (fast predicate pushdown)
    bars = equity_reader.get_bars(
        symbol=symbol,
        bar_size="1 day",
        start_date=date.today() - timedelta(days=10),
        end_date=date.today(),
        use_pyarrow=True  # Use PyArrow for large scans
    )
    
    print(f"\n{symbol} bars (last 10 days): {len(bars)} rows")
    print(f"Query method: PyArrow with predicate pushdown")
    
    bars.head(10)

#### Get summary for all symbols

In [ ]:
summary = equity_reader.get_symbols_summary(bar_size="1 day")
print("\nSymbol summary (aggregated with DuckDB):\n")
summary

#### Count bars efficiently

In [ ]:
if symbols:
    # Count for specific symbol
    count = equity_reader.count(symbol=symbol, bar_size="1 day")
    print(f"\nTotal bars for {symbol}: {count:,}")
    
    # Count all bars
    total_count = equity_reader.count()
    print(f"Total bars (all symbols): {total_count:,}")

### OptionChainSnapshotReader

In [ ]:
# Initialize option chain reader
option_chain_reader = OptionChainSnapshotReader("../data", "options")

print("✓ OptionChainSnapshotReader initialized")

#### Get available snapshots

In [ ]:
# Check if we have any option snapshots
snapshots = option_chain_reader.get_available_snapshots("SPY")

if snapshots:
    print(f"\nAvailable snapshots for SPY: {len(snapshots)} dates")
    for snapshot_date in sorted(snapshots, reverse=True)[:5]:
        print(f"  - {snapshot_date}")
else:
    print("\n⚠️  No option snapshots found. Run the option snapshot cell in 01_quickstart.ipynb")

#### Get available expirations

In [ ]:
if snapshots:
    latest_snapshot = max(snapshots)
    
    # Get all expirations
    expirations = option_chain_reader.get_available_expirations(
        underlying="SPY",
        as_of=latest_snapshot,
    )
    
    print(f"\nExpirations available on {latest_snapshot}:")
    for exp in expirations[:10]:  # Show first 10
        dte = (exp - latest_snapshot).days
        print(f"  - {exp} (DTE: {dte})")

#### Get strikes for an expiration

In [ ]:
if snapshots and expirations:
    # Get strikes for the first expiration
    strikes = option_chain_reader.get_strikes_for_expiry(
        underlying="SPY",
        as_of=latest_snapshot,
        expiry=expirations[0],
    )
    
    print(f"\nStrikes for {expirations[0]}: {len(strikes)} strikes")
    print(f"Range: ${min(strikes):.2f} - ${max(strikes):.2f}")
    print(f"\nFirst 10 strikes: {strikes[:10]}")

## 4. Advanced PyArrow Queries

PyArrow provides direct Parquet access with efficient predicate pushdown.

### Load Parquet dataset with PyArrow

In [ ]:
# Load dataset with Hive partitioning
dataset_path = data_dir / "stocks"

if dataset_path.exists():
    dataset = ds.dataset(
        dataset_path,
        format="parquet",
        partitioning="hive"
    )
    
    print(f"✓ Loaded PyArrow dataset from {dataset_path}")
    print(f"\nSchema:")
    print(dataset.schema)
else:
    print(f"⚠️  Dataset not found: {dataset_path}")

### Query with filters (predicate pushdown)

In [ ]:
if dataset_path.exists() and symbols:
    # Create filter expression
    filter_expr = (
        (pc.field("symbol") == symbol) &
        (pc.field("date") >= (date.today() - timedelta(days=7)).isoformat())
    )
    
    # Scan with filter (predicate pushdown = only reads matching partitions)
    scanner = dataset.scanner(
        filter=filter_expr,
        columns=["symbol", "time", "date", "open", "high", "low", "close", "volume"]
    )
    
    # Convert to pandas
    table = scanner.to_table()
    df = table.to_pandas()
    
    print(f"\nPyArrow query results: {len(df)} rows")
    print(f"Filter: symbol={symbol}, date >= last 7 days")
    print(f"\nNote: Only partition directories matching the filter were read!")
    
    df.head(10)

### Show partitions (directories)

In [ ]:
if dataset_path.exists():
    # List partition directories
    partitions = []
    for date_dir in sorted((dataset_path / "equity_bars_backfill").glob("date=*"))[:5]:
        date_value = date_dir.name.split("=")[1]
        for symbol_dir in sorted(date_dir.glob("symbol=*")):
            symbol_value = symbol_dir.name.split("=")[1]
            parquet_files = list(symbol_dir.glob("*.parquet"))
            partitions.append({
                "date": date_value,
                "symbol": symbol_value,
                "parquet_files": len(parquet_files)
            })
    
    if partitions:
        print("\nPartition structure (first 10):")
        partition_df = pd.DataFrame(partitions)
        partition_df.head(10)

## 5. Performance Comparison

Compare query performance: DuckDB vs PyArrow vs Reader API

In [ ]:
import time

if symbols:
    symbol_to_test = symbols[0]
    
    # Test 1: DuckDB query
    start = time.time()
    df_duckdb = conn.execute(f"""
        SELECT *
        FROM parquet_scan('../data/stocks/**/*.parquet', hive_partitioning=true)
        WHERE symbol = '{symbol_to_test}'
    """).df()
    duckdb_time = time.time() - start
    
    # Test 2: PyArrow query
    start = time.time()
    filter_expr = pc.field("symbol") == symbol_to_test
    scanner = dataset.scanner(filter=filter_expr)
    df_pyarrow = scanner.to_table().to_pandas()
    pyarrow_time = time.time() - start
    
    # Test 3: Reader API (uses PyArrow internally)
    start = time.time()
    df_reader = equity_reader.get_bars(
        symbol=symbol_to_test,
        bar_size="1 day",
        use_pyarrow=True
    )
    reader_time = time.time() - start
    
    print(f"\n⏱️  Performance Comparison ({symbol_to_test}, {len(df_duckdb)} rows):\n")
    print(f"  DuckDB:     {duckdb_time*1000:.2f} ms")
    print(f"  PyArrow:    {pyarrow_time*1000:.2f} ms")
    print(f"  Reader API: {reader_time*1000:.2f} ms")
    
    print(f"\n💡 Notes:")
    print(f"  - DuckDB: Best for SQL analytics and aggregations")
    print(f"  - PyArrow: Best for large scans with simple filters")
    print(f"  - Reader API: Most convenient, uses PyArrow internally")
    print(f"  - All methods benefit from Hive partitioning (predicate pushdown)")

## Summary

In this notebook, you learned:

1. ✓ How Parquet files are stored with Hive-style partitioning
2. ✓ Querying Parquet with DuckDB SQL
3. ✓ Using Reader Repositories for convenient Python API
4. ✓ Direct PyArrow access for advanced queries
5. ✓ Performance characteristics of different query methods

### Best Practices

**Use DuckDB when:**
- You need SQL analytics (GROUP BY, window functions, etc.)
- Working with multiple tables/joins
- Generating reports or summaries

**Use PyArrow when:**
- Reading large datasets with simple filters
- Need maximum performance for scans
- Working programmatically in Python

**Use Reader Repositories when:**
- You want a simple Python API
- Common queries (get_bars, get_available_symbols)
- Don't want to write SQL

### Partitioning Benefits

Hive-style partitioning by date and symbol means:
- ✓ Queries filter partitions before reading files
- ✓ Only matching directories are scanned
- ✓ Massive performance improvement for filtered queries
- ✓ Works seamlessly with all query methods

Example: `WHERE symbol = 'AAPL' AND date >= '2025-10-01'`
- Only reads: `data/stocks/*/date=2025-10-*/symbol=AAPL/*.parquet`
- Skips all other partition directories